# Pilot runs — slm-audio-evidence (Colab)

Runtime → Change runtime type → **T4 GPU**. Затем — ячейки сверху вниз (ячейка 4 проверяет GPU автоматически и падает с понятной ошибкой, если досталась несовместимая карта).

По умолчанию инференс НЕ перезапускается: `responses.jsonl` и ручная разметка `manual-M1` уже закоммичены в `results/`. Ячейки ниже сразу переходят к прогону LLM-судьи и сравнению с ней — аудио для этого не нужно (судья читает только текстовый транскрипт из манифеста).

Нужно пересчитать инференс с нуля (новые аудио/модели/промпты)? Поставь `RUN_INFERENCE = True` в соответствующей ячейке — тогда понадобится `pilot_audio.zip`, загруженный через Files panel (левая панель, drag & drop) в `/content/`.

Нет доступа на чтение/клонирование репозитория? Ниже (ячейка 2) есть флаг `USE_LOCAL_ZIP` — поставь `True` и загрузи zip репозитория через Files panel вместо `git clone`.

In [ ]:
!nvidia-smi -L

In [ ]:
import subprocess

def sh(cmd: str) -> None:
    subprocess.run(cmd, shell=True, check=True)

# TEMP while testing on the fork, pre-merge (see docs/decisions.md 2026-07-15): clone the
# fork's branch, not origin/main -- no push/PR yet. Switch back to REPO_URL of
# https://github.com/ladnlav/slm-audio-evidence.git and REPO_BRANCH="main" once merged.
REPO_URL = "https://github.com/PolinaSh-main/slm-audio-evidence.git"
REPO_BRANCH = "m3/llm-judge"

# False (default): git clone from GitHub (needs read access to the repo).
# True: no repo access -- unzip a manually uploaded copy instead (see REPO_ZIP_PATH below).
USE_LOCAL_ZIP = False
REPO_ZIP_PATH = "/content/slm-audio-evidence.zip"

if USE_LOCAL_ZIP:
    sh(f"mkdir -p slm-audio-evidence && unzip -q -o {REPO_ZIP_PATH} -d slm-audio-evidence")
else:
    sh(f"git clone --branch {REPO_BRANCH} {REPO_URL}")
%cd slm-audio-evidence

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU visible -- Settings (right panel) -> Accelerator, pick GPU T4 x2, "
        "then Restart session and rerun from cell 2."
    )

name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
print(f"GPU: {name} (compute capability {capability[0]}.{capability[1]})")
if capability < (7, 0):
    raise RuntimeError(
        f"{name} (compute capability {capability[0]}.{capability[1]}) is too old for the "
        "preinstalled PyTorch build (needs >= 7.0, e.g. T4/V100/A100). Seen in practice: "
        "P100 (Pascal, 6.0) fails here. Settings (right panel) -> Accelerator -> switch to "
        "GPU T4 x2, then Restart session and rerun from cell 2."
    )

In [ ]:
# True only to regenerate responses.jsonl from scratch (new audio/models/prompts).
# False (default): skip audio + inference entirely -- responses.jsonl and the
# manual-M1 human labels are already committed in results/.
RUN_INFERENCE = False

# Only read if RUN_INFERENCE is True. Upload pilot_audio.zip to /content first
# (drag & drop into the Files panel, left sidebar).
AUDIO_ZIP_PATH = "/content/pilot_audio.zip"

In [ ]:
if RUN_INFERENCE:
    sh(f'unzip -q -o {AUDIO_ZIP_PATH} -d .')
    sh('ls data/audio/spoken_squad_test | head -3')
    import json, os
    rows = [json.loads(l) for l in open('data/manifests/pilot.jsonl', encoding='utf-8')]
    miss = [r['id'] for r in rows if not os.path.exists(r['audio_path'])]
    print(len(rows), 'items,', len(miss), 'missing audio')
    print(miss[:5])
else:
    print('RUN_INFERENCE=False -- skipping audio upload/check (responses.jsonl already in results/).')

In [ ]:
# Run 1 (the headline): Qwen2-Audio, plain prompt. First run also downloads the weights (~15-30 min).
if RUN_INFERENCE:
    sh('python -m src.inference --model qwen2audio --strategy plain --data data/manifests/pilot.jsonl --out results/')
else:
    print('RUN_INFERENCE=False -- skipping.')

In [ ]:
# Run 2: Qwen2-Audio, IDK prompt
if RUN_INFERENCE:
    sh('python -m src.inference --model qwen2audio --strategy s1_idk --data data/manifests/pilot.jsonl --out results/')
else:
    print('RUN_INFERENCE=False -- skipping.')

In [ ]:
# IMPORTANT before runs 3-4: free VRAM from Qwen2-Audio -- Runtime -> Restart runtime,
# then re-run cells 2 (clone/cd), 3 (pip install), 4 (flags) and, if RUN_INFERENCE, cell 5 (unzip)
# -- then continue here.
if RUN_INFERENCE:
    sh('python -m src.inference --model cascade --strategy plain --data data/manifests/pilot.jsonl --out results/')
else:
    print('RUN_INFERENCE=False -- skipping.')

In [ ]:
# Run 4: cascade, IDK prompt
if RUN_INFERENCE:
    sh('python -m src.inference --model cascade --strategy s1_idk --data data/manifests/pilot.jsonl --out results/')
else:
    print('RUN_INFERENCE=False -- skipping.')

In [ ]:
# Zip and download everything produced so far (run after EACH finished run -- do not wait for all four)
if RUN_INFERENCE:
    sh("zip -q -r results_runs.zip results -x '*.gitkeep'")
    from google.colab import files
    files.download('results_runs.zip')
else:
    print('RUN_INFERENCE=False -- nothing new to zip.')

## LLM-судья (категория B) vs ручная разметка manual-M1
`results/<run_id>/responses.jsonl` уже есть (см. выше). Категория B (label=answer) в них размечена людьми вручную (`results/<run_id>/responses_judged.jsonl`, `judge: "manual-M1"`, committed — см. docs/decisions.md) — это уже готовая истина, повторную слепую разметку делать не нужно.

Судья пишет в **отдельную** папку `results/<run_id>/llm_audit_<версия промпта>/`, а не поверх `responses_judged.jsonl` — тот файл с ручной разметкой нельзя перезаписывать, он невоспроизводим.

Запускаем ОДНИМ Python-процессом (не 4 отдельных вызова) — модель-судья (`Qwen/Qwen3-8B`, int8) грузится один раз и переиспользуется на все 4 прогона: быстрее и без риска VRAM-утечек между процессами, как при повторных инференс-прогонах выше. Если увидите OOM — перезапустите сессию/kernel и заново выполните ячейки клонирования, установки зависимостей и эту.

**Промпт**: `judge_v1.txt` — единственная используемая версия (92% на Qwen3-8B, см. docs/decisions.md). `judge_v2.txt` (v1 + анти-verbosity предложение) протестирован и **отклонён**: просело до 89% — при коротком бюджете токенов и выключенном thinking судья вместо проверки фактов начал банить сам факт многословности, ломая правильные развёрнутые ответы чаще, чем чинил неправильные (см. docs/decisions.md, judge_v2 post-mortem).

**Рассуждение перед вердиктом**: с 2026-07-15 `LocalHFJudge` умеет генерировать с `enable_thinking=True` (родной `<think>...</think>` Qwen3) — у судьи есть место реально сверить факты перед ответом, а не паттерн-матчить по длине ответа. Генерация останавливается сама, как только после `</think>` появляется вердикт-слово (`_StopOnVerdict` в `local_hf.py`), а не всегда тянется до `max_new_tokens=512` — но thinking всё равно кратно медленнее, чем прямой ответ без рассуждения (`enable_thinking=False`, было раньше).

**Скорость на реальном масштабе**: thinking на каждом элементе — потратить 20-40+ минут даже на пилоте (~90 B-примеров); на 1000+ примерах и нескольких прогонах это уже недопустимо. `JUDGE_MODE` в ячейке ниже:
- `"tiered"` (по умолчанию, для реальных прогонов) — быстрый no-think проход на 100% данных, thinking-проход только на ответах, которые выглядят подозрительно многословными относительно gold (response/gold по словам ≥3×, см. `src/judges/tiered.py`) — ровно тот паттерн, где thinking реально помогает. Экономит порядка 80-90% времени судьи почти без потери точности апгрейда.
- `"dev_subset"` — фиксированный набор ~23 примеров (известные сложные случаи из прошлых аудитов + контрольные лёгкие, `src/judges/dev_subset.py`) — проверить новый промпт/модель за 1-2 минуты, не платя за полный прогон каждый раз.
- `"full"` — thinking на 100% данных без каскада (как было раньше). Годится для пилота, не годится для основной фазы.

`judge_name` у каждого режима свой (`llm-qwen3-8b-v1` / `-v2` / `tiered-llm-qwen3-8b`), так что кеш (`judge_cache.jsonl`) никогда не путает вердикты разных режимов между собой.

**Кэш весов судьи между сессиями**: `Qwen3-8B` (~16 ГБ) по умолчанию качается в `~/.cache/huggingface`, который живёт вне `/kaggle/working` и пропадает при пересоздании сессии. Ячейка ниже кладёт кэш в `/kaggle/working/hf_cache` и, если он там уже есть (например, восстановлен из прикреплённого Kaggle Dataset), переиспользует его вместо скачивания. Чтобы закэшировать один раз и не качать в следующих сессиях: после первого успешного прогона сохрани `/kaggle/working/hf_cache` как новый Kaggle Dataset (Output tab у ноутбука -> New Dataset), в следующей сессии прикрепи его через Add Data и поправь путь `_prebuilt_cache` ниже под реальное имя датасета.

In [ ]:
import os

HF_CACHE_DIR = "/kaggle/working/hf_cache"
os.environ.setdefault("HF_HOME", HF_CACHE_DIR)

# Reuse a pre-downloaded cache if you saved one from a previous session (see markdown above)
# instead of re-downloading Qwen3-8B (~16 GB) from scratch.
_prebuilt_cache = "/kaggle/input/qwen-judge-cache/hf_cache"  # adjust to your dataset's actual path
if os.path.isdir(_prebuilt_cache) and not os.path.isdir(HF_CACHE_DIR):
    sh(f"cp -r {_prebuilt_cache} {HF_CACHE_DIR}")
    print(f"Reused cached weights from {_prebuilt_cache} -- no download needed.")

from src.judges import build_judge
from src.judges.dev_subset import DEV_SUBSET
from src.run_eval import run_evaluation

RUNS = [
    "qwen2audio_plain_20260712",
    "qwen2audio_s1_idk_20260712",
    "cascade_plain_20260712",
    "cascade_s1_idk_20260712",
]

# judge_v1.txt only -- judge_v2.txt (anti-verbosity sentence) was tested and rejected
# (92% -> 89%, see docs/decisions.md). Output dir is named after the prompt, so switching
# this and rerunning does NOT overwrite another version's results -- they stay around to compare.
JUDGE_PROMPT = "judge_v1.txt"
AUDIT_SUBDIR = "llm_audit_" + JUDGE_PROMPT.replace("judge_", "").replace(".txt", "")  # -> llm_audit_v1

# "tiered" (default, for real-scale runs) / "dev_subset" (fast iteration, ~23 fixed items) /
# "full" (thinking on every item -- fine for the ~90-item pilot, not for 1000+). See markdown above.
JUDGE_MODE = "tiered"

judge_backend = "tiered" if JUDGE_MODE == "tiered" else "local"
judge = build_judge(judge_backend, prompt_name=JUDGE_PROMPT)  # Qwen3-8B, int8 -- loads once (~2-4 min)
for run_id in RUNS:
    subset_ids = set(DEV_SUBSET.get(run_id, [])) if JUDGE_MODE == "dev_subset" else None
    run_evaluation(
        manifest_path="data/manifests/pilot.jsonl",
        responses_path=f"results/{run_id}/responses.jsonl",
        out_dir=f"results/{run_id}/{AUDIT_SUBDIR}",  # separate dir -- never touches committed manual-M1
        judge=judge,
        subset_ids=subset_ids,
    )

if hasattr(judge, "escalated_count"):  # only TieredJudge tracks this
    print(f"[tiered] escalated {judge.escalated_count}/{judge.total_count} items to the slow (thinking) pass")

Сравнить свежие вердикты судьи с manual-M1 (гейт согласия ROLE_M3: 80%; если ниже — не публиковать LLM-judge цифры, см. вывод команды). Использует тот же `AUDIT_SUBDIR` и отдельный `--out` на промпт, что и ячейка выше — сравнения `judge_v1` и `judge_v2` не затирают друг друга:

In [ ]:
sh(
    'python scripts/audit_judge.py compare '
    '--judged "results/*/responses_judged.jsonl" '
    f'--judge-subdir {AUDIT_SUBDIR} '
    f'--out results/judge_audit_{AUDIT_SUBDIR.replace("llm_audit_", "")}'
)

In [ ]:
# Собрать вердикты судьи + отчёт о согласии в zip (оба промпта, если гоняли оба)
!zip -q -r judge_audit.zip results/*/llm_audit_* results/judge_audit_* -x '*.gitkeep'
print('judge_audit.zip written -- grab it from the notebook Output tab '
      '(Kaggle persists everything under /kaggle/working).')